In [1]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### E14: motor frequency x severity on the v6 structure (Oct 24, notebook 27) — rule fixed before running
- Part 1's final motor structure (E11, v12: +0.0025 validation D2) was never carried into Part 2, which uses the v11 single Tweedie GLM.
- Design: Poisson frequency and Gamma severity GLMs, both fitted on the developed table (as v6 motor, developed relativities);
  same features, bands, target (2025 level, capped, net of excess), completeness weights; ridge for each part chosen inside the
  developed data (fit 2021-23, choose on 2024) from Part 1's list; one emerging level factor on 2023. Validate emerging 2024.
- Rule: adopt if motor 2024 D2 >= base + 0.0005 and neither decile A/E moves more than 0.1 further from 1 (base 0.0373, low 0.794, top 0.894).
- Expected: +0.000 to +0.003. If adopted, the new-business load is re-measured before any submission.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
assert "### E14: motor frequency x severity on the v6" not in d, "already recorded"
a = "## Assumptions register"; assert d.count(a) == 1
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [2]:
import os, sys, copy, time; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA")); sys.path.insert(0, str(Path.home()/"IAA"/"tools"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.evaluate import evaluate
from exp_freqsev import ALPHAS, _glm, _choose_alpha

cfg = copy.deepcopy(PART2); line = "motor"
T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])
d = P.build_line(line, cfg, claims)
tr0 = d["tr"]; tr, va = tr0[tr0.year == 2023], tr0[tr0.year == 2024]

dev = P.developed_table(line, cfg).copy()
c1 = T.load_claims(PART1["data_dir"]); p1 = T.load_policies(line, "train", PART1["data_dir"])
mc1 = T.adjust_claims(T.match_claims(c1, p1, line, True, cap=None), PART1["adj"][line])
dev["n_claims"] = dev.row_id.map(mc1[mc1.net_adj > 0].groupby("row_id").size()).fillna(0).astype(float)
f = [c for c in d["feats"] if c in dev.columns]
P.log(f"developed rows {len(dev):,}, claim rows {(dev.n_claims > 0).sum():,}, features {len(f)}")

# frequency x severity on developed data, ridge per part chosen on developed 2024
inner, hold = dev[dev.year < dev.year.max()], dev[dev.year == dev.year.max()]
yf, wf = (lambda x: x.n_claims / x.weight), (lambda x: x.weight)
ys, ws = (lambda x: x.target / x.n_claims), (lambda x: x.n_claims)
kw = {"glm": {line: PART1["glm"][line]}}
af = _choose_alpha(kw, line, 1.0, inner, hold, f, yf, wf)
as_ = _choose_alpha(kw, line, 2.0, inner[inner.n_claims > 0], hold[hold.n_claims > 0], f, ys, ws)
P.log(f"alpha chosen on developed 2024: frequency {af}, severity {as_}")
pos = dev[dev.n_claims > 0]
gf = _glm(kw, line, 1.0, af).fit(dev[f], yf(dev), wf(dev))
gs = _glm(kw, line, 2.0, as_).fit(pos[f], ys(pos), ws(pos))
pr = lambda X: gf.predict_rate(X[f]) * gs.predict_rate(X[f])
lvl = tr.target.sum() / (pr(tr) * tr.weight).sum()
fs = pr(va) * lvl

base, _ = P.predict_rate(line, cfg, tr, va, d["feats"])
def m(rate):
    r = evaluate(va.target, np.asarray(rate) * va.weight.to_numpy(), va.weight, show=False); lf = r["lift"]["A/E"]
    return {"D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1]}
b, n = m(base), m(fs)
print(pd.DataFrame({"base (Tweedie, developed)": b, "freq x sev (developed)": n}).T.round(4).to_string())
assert abs(b["D2"] - 0.0373) < 0.0005, "base differs from today's fold"
gain = n["D2"] - b["D2"]
tails = all(abs(n[k] - 1) <= abs(b[k] - 1) + 0.1 for k in ["low", "top"])
print(f"\nD2 {gain:+.4f} | tails {'ok' if tails else 'FAIL'} | emerging level x{lvl:.4f} -> {'ADOPT' if gain >= 0.0005 and tails else 'not adopted'}")

20:58:56 motor: developed training table 979,968 rows
20:58:56 developed rows 979,968, claim rows 98,089, features 42
21:01:08 alpha chosen on developed 2024: frequency 0.001, severity 0.01
                               D2     A/E     low     top
base (Tweedie, developed)  0.0373  1.0764  0.7940  0.8936
freq x sev (developed)     0.0364  1.0419  0.7664  0.8463

D2 -0.0009 | tails ok | emerging level x0.7652 -> not adopted


In [3]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])
d = P.build_line("buildings", cfg, claims)
tr = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id")
ALPHAS = [0.5, 2, 10, 50, 200, 1000]          # the Oct 23 list
grp = (pd.util.hash_pandas_object(tr.policy_id.astype(str), index=False) % 5).to_numpy()

def cv(df, g):
    out = {}
    for a in ALPHAS:
        parts = []
        for k in range(5):
            fit, hold = df[g != k], df[g == k]
            m = BandedTweedieGLM(**dict(cfg["glm"]["buildings"], alpha=a)).fit(fit[d["feats"]], fit.target / fit.weight, fit.weight)
            parts.append(pd.DataFrame({"a": hold.target, "p": m.predict_rate(hold[d["feats"]]) * hold.weight, "w": hold.weight}))
        x = pd.concat(parts); out[a] = evaluate(x.a, x.p, x.w, show=False)["D2"]
    return pd.Series(out)

m23 = (tr.year == 2023).to_numpy()
res = pd.DataFrame({"CV D2, 2023 only (as Oct 23)": cv(tr[m23], grp[m23]), "CV D2, 2023+2024 (final training)": cv(tr, grp)})
print(res.round(4).to_string())
print("\nchosen alpha:", res.idxmax().to_dict())

21:02:47 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:02:47 buildings: fire priced separately at 97.05 per exposure (developed)
21:02:48 buildings: own GLM for ['flood']; main GLM on the other perils
        CV D2, 2023 only (as Oct 23)  CV D2, 2023+2024 (final training)
0.5                          -0.1337                            -0.0432
2.0                          -0.0203                             0.0056
10.0                          0.0012                             0.0082
50.0                         -0.0003                             0.0019
200.0                        -0.0017                            -0.0003
1000.0                       -0.0021                            -0.0010

chosen alpha: {'CV D2, 2023 only (as Oct 23)': 10.0, 'CV D2, 2023+2024 (final training)': 10.0}


In [4]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### E24: buildings other perils on combined data, re-run on the corrected target (Oct 24, notebook 28) — rule fixed before running (decided by Woojin)
- E17b / E18 (old target): combined training raised buildings 2024 D2 by about +0.02-0.03 but failed the lowest decile (rule b: 0.46 vs 0.89).
- Notebook 27 (AI 111) showed buildings decile cost A/E is dominated by single large losses (A/E without each decile's largest loss 0.28-0.88),
  so a one-decile, one-year cost A/E cannot separate mispricing from chance.
- Candidates: developed weight 1 / 0.3 / 0.1 (E18 list); flood GLM and the flat fire rate unchanged; level on emerging 2023; validate 2024.
- Rule: adopt the best candidate if 2024 D2 (all perils) >= current + 0.005 AND, with each decile's largest loss removed (modelled part, rule b),
  neither the lowest nor the highest decile A/E moves more than 0.1 further from 1 than the current model. Rule b without removal reported too.
- The tail reading changed before running, because of the large-loss evidence above, not because of any E24 result.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
assert "### E24:" not in d, "already recorded"
a = "## Assumptions register"; assert d.count(a) == 1
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [5]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"]); line = "buildings"
d = P.build_line(line, cfg, claims); feats = d["feats"]
tr, va = d["tr"][d["tr"].year == 2023], d["tr"][d["tr"].year == 2024]
rate_base, _ = P.predict_rate(line, cfg, tr, va, feats)
main_cur = BandedTweedieGLM(**cfg["glm"][line]).fit(tr[feats], tr.target / tr.weight, tr.weight).predict_rate(va[feats])

dev = P.developed_table(line, cfg); f = [c for c in feats if c in dev.columns]; kw = PART1["glm"][line]
rates = {"current (emerging only)": np.asarray(rate_base)}
for wd in [1.0, 0.3, 0.1]:
    both = pd.concat([dev[f + ["target", "weight"]].assign(market="developed", weight=dev.weight * wd),
                      tr[f + ["target", "weight"]].assign(market="emerging")], ignore_index=True)
    g = BandedTweedieGLM(**kw).fit(both[f + ["market"]], both.target / both.weight, both.weight)
    pr = lambda X: g.predict_rate(X[f].assign(market="emerging"))
    lvl = tr.target.sum() / (pr(tr) * tr.weight).sum()
    rates[f"combined, dev weight {wd}"] = np.asarray(rate_base) - np.asarray(main_cur) + pr(va) * lvl

w = va.weight.to_numpy(); act_all = va.target_all.to_numpy(); act_mod = va.target_modelled.to_numpy()
def deciles(rate, act):
    o = np.argsort(rate, kind="stable"); cw = np.cumsum(w[o]) / w.sum(); dec = np.minimum((cw * 10).astype(int), 9)
    pv, a = (rate * w)[o], act[o]; out = []
    for k in (0, 9):
        s = dec == k; out += [a[s].sum() / pv[s].sum(), (a[s].sum() - a[s].max()) / pv[s].sum()]
    return out
rows = []
for name, r in rates.items():
    d2 = evaluate(pd.Series(act_all), (r + d["sep_rate"]) * w, pd.Series(w), show=False)["D2"]
    lo, lo_x, hi, hi_x = deciles(r, act_mod)
    rows.append({"model": name, "D2 all perils": d2, "low (rule b)": lo, "top (rule b)": hi,
                 "low, largest removed": lo_x, "top, largest removed": hi_x})
t = pd.DataFrame(rows).set_index("model"); print(t.round(4).to_string())
b = t.iloc[0]; assert abs(b["D2 all perils"] - 0.0684) < 0.0005, "base differs from today's fold"
for name, r in t.iloc[1:].iterrows():
    gain = r["D2 all perils"] - b["D2 all perils"]
    tails = all(abs(r[k] - 1) <= abs(b[k] - 1) + 0.1 for k in ["low, largest removed", "top, largest removed"])
    print(f"{name}: D2 {gain:+.4f} | tails (largest removed) {'ok' if tails else 'FAIL'} -> {'passes' if gain >= 0.005 and tails else 'fails'}")

21:03:37 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:03:37 buildings: fire priced separately at 97.05 per exposure (developed)
21:03:38 buildings: own GLM for ['flood']; main GLM on the other perils
21:03:43 buildings: developed training table 707,726 rows
                          D2 all perils  low (rule b)  top (rule b)  low, largest removed  top, largest removed
model                                                                                                          
current (emerging only)          0.0684        0.6154        1.0215                0.5149                0.8889
combined, dev weight 1.0         0.0950        0.3698        0.9845                0.3200                0.8532
combined, dev weight 0.3         0.0952        0.4405        0.9474                0.3873                0.8175
combined, dev weight 0.1         0.0937        0.4264        0.9404                0.3712                0.8117
combined, dev weight 1.0: D2 +0.0266 | 

In [9]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
assert "**E14 result:**" not in d and "**E24 result:**" not in d and "Buildings ridge re-checked" not in d, "already recorded"

# E14: fix notebook number, append result under its block
h14 = "### E14: motor frequency x severity on the v6 structure (Oct 24, notebook 27)"
assert d.count(h14) == 1; d = d.replace(h14, h14.replace("notebook 27", "notebook 28"))
a14 = "- Expected: +0.000 to +0.003. If adopted, the new-business load is re-measured before any submission.\n"
assert d.count(a14) == 1
d = d.replace(a14, a14 + "- **E14 result:** ridge chosen on developed 2024: frequency 0.001, severity 0.01. Motor 2024 D2 0.0373 -> 0.0364 (-0.0009); "
              "low 0.794 -> 0.766, top 0.894 -> 0.846 (within the limit); emerging level x0.765. **Not adopted.** Part 1's +0.0025 does not carry over: "
              "the relativities already come from 98k developed claim rows.\n\n"
              "### Buildings ridge re-checked on the final training data (Oct 24, notebook 28)\n"
              "- Concern: sklearn's penalty is on the mean deviance, so an alpha chosen on one year (2023 CV) acts about twice as strongly on two years.\n"
              "- Same rule as Oct 23 (0.5/2/10/50/200/1000, 5-fold CV by policy): 2023 only picks 10 (CV D2 0.0012); 2023+2024 also picks 10 "
              "(0.0082; next 2: 0.0056). **No change.**\n")

# E24: append result after the last line of its block
a24 = "- The tail reading changed before running, because of the large-loss evidence above, not because of any E24 result.\n"
assert d.count(a24) == 1
d = d.replace(a24, a24 + "- Rule written 21:03:33, run started 21:03:37 (Oct 24).\n"
              "- **E24 result:** 2024 D2 (all perils) current 0.0684 -> dev weight 1 / 0.3 / 0.1: 0.0950 / 0.0952 / 0.0937 (+0.027 / +0.027 / +0.025). "
              "Lowest decile A/E with each decile's largest loss removed: 0.515 -> 0.320 / 0.387 / 0.371 (0.13-0.20 further from 1; limit 0.1); "
              "top within the limit. Rule b without removal: 0.615 -> 0.370 / 0.441 / 0.426. **Not adopted.**\n"
              "- Fourth time the same result (E17b, E18 twice, E24), now with the largest loss removed: developed relativities over-price the safest "
              "emerging homes (structural, not chance). Cost of protecting the safest decile, for the report: about +0.027 buildings validation D2 forgone.\n")
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [10]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### E25: buildings combined base + emerging corrections on selected features (Oct 24, notebook 28) — rule fixed before running
- Designed after E24 failed the lowest decile (disclosed). Aim: keep the combined-data gain (+0.027) and fix the safest decile.
- Base o: other-perils GLM on combined data (developed weight 1, pipeline default; E24's best weight 0.3 is not used because it was picked on 2024),
  level on emerging 2023. Flood GLM and the flat fire rate unchanged.
- Feature selection on emerging 2023 only (2024 not used): for each feature, a one-feature GLM on top of o (offset via y/o with weight w*sqrt(o),
  exact for Tweedie p = 1.5, as E21); 5-fold CV inside 2023 grouped by policy; keep features with CV D2 gain >= +0.001, at most the top 5.
  Developed-absent features (e.g. gender) are candidates too. One correction GLM on the selected features, alpha 10, fitted on 2023.
- Rule (same as E24): adopt if 2024 D2 (all perils) >= current + 0.005 AND, with each decile's largest loss removed (modelled part, rule b),
  neither tail moves more than 0.1 further from 1 than the current model. 2024 is used once.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
assert "### E25:" not in d, "already recorded"
a = "## Assumptions register"; assert d.count(a) == 1
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [12]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"]); line = "buildings"
d = P.build_line(line, cfg, claims); feats = d["feats"]
tr0 = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id")
tr, va = tr0[tr0.year == 2023].reset_index(drop=True), tr0[tr0.year == 2024].reset_index(drop=True)

rate_base, _ = P.predict_rate(line, cfg, tr, va, feats); rate_base = np.asarray(rate_base)
mc_va = BandedTweedieGLM(**cfg["glm"][line]).fit(tr[feats], tr.target / tr.weight, tr.weight).predict_rate(va[feats])

# base o: combined data, developed weight 1, level on emerging 2023
dev = P.developed_table(line, cfg); f = [c for c in feats if c in dev.columns]
both = pd.concat([dev[f + ["target", "weight"]].assign(market="developed"),
                  tr[f + ["target", "weight"]].assign(market="emerging")], ignore_index=True)
g = BandedTweedieGLM(**PART1["glm"][line]).fit(both[f + ["market"]], both.target / both.weight, both.weight)
pr = lambda X: g.predict_rate(X[f].assign(market="emerging"))
lvl = tr.target.sum() / (pr(tr) * tr.weight).sum()
o_tr, o_va = np.asarray(pr(tr)) * lvl, np.asarray(pr(va)) * lvl

def corr_fit(X, r, w, o, cols):
    return BandedTweedieGLM(power=1.5, alpha=10, n_bands=10, drop=None, balance=True).fit(X[cols], r / o, w * np.sqrt(o))
def d2(a, pv, w): return evaluate(pd.Series(np.asarray(a)), np.asarray(pv), pd.Series(np.asarray(w)), show=False)["D2"]

# feature screening inside 2023 only
grp = (pd.util.hash_pandas_object(tr.policy_id.astype(str), index=False) % 5).to_numpy()
r_tr, w_tr = (tr.target / tr.weight).to_numpy(), tr.weight.to_numpy()
def cv_gain(cols):
    base_p, new_p = np.zeros(len(tr)), np.zeros(len(tr))
    for k in range(5):
        fi, ho = grp != k, grp == k
        k0 = tr.target[fi].sum() / (o_tr[fi] * w_tr[fi]).sum(); base_p[ho] = o_tr[ho] * k0 * w_tr[ho]
        c = corr_fit(tr[fi], r_tr[fi], w_tr[fi], o_tr[fi], cols)
        k1 = tr.target[fi].sum() / (o_tr[fi] * c.predict_rate(tr[fi][cols]) * w_tr[fi]).sum()
        new_p[ho] = o_tr[ho] * c.predict_rate(tr[ho][cols]) * k1 * w_tr[ho]
    return d2(tr.target, new_p, w_tr) - d2(tr.target, base_p, w_tr)
scr = pd.Series({c: cv_gain([c]) for c in feats}).sort_values(ascending=False)
print("CV D2 gain inside 2023 (top 12):"); print(scr.head(12).round(4).to_string())
sel = [c for c in scr.index if scr[c] >= 0.001][:5]
print("selected:", sel)

# E25 rate on 2024 (other-perils part replaced; flood and fire unchanged)
if sel:
    c = corr_fit(tr, r_tr, w_tr, o_tr, sel)
    k = tr.target.sum() / (o_tr * c.predict_rate(tr[sel]) * w_tr).sum()
    main_new = o_va * c.predict_rate(va[sel]) * k
else:
    main_new = o_va
rates = {"current (emerging only)": rate_base,
         "combined, dev weight 1 (E24)": rate_base - np.asarray(mc_va) + o_va,
         "E25 combined + corrections": rate_base - np.asarray(mc_va) + main_new}

w = va.weight.to_numpy(); act_all = va.target_all.to_numpy(); act_mod = va.target_modelled.to_numpy()
def deciles(rate, act):
    o = np.argsort(rate, kind="stable"); cw = np.cumsum(w[o]) / w.sum(); dec = np.minimum((cw * 10).astype(int), 9)
    pv, a = (rate * w)[o], act[o]; out = []
    for kk in (0, 9):
        s = dec == kk; out += [a[s].sum() / pv[s].sum(), (a[s].sum() - a[s].max()) / pv[s].sum()]
    return out
rows = []
for name, r in rates.items():
    lo, lo_x, hi, hi_x = deciles(r, act_mod)
    rows.append({"model": name, "D2 all perils": d2(act_all, (r + d["sep_rate"]) * w, w), "low (rule b)": lo, "top (rule b)": hi,
                 "low, largest removed": lo_x, "top, largest removed": hi_x})
t = pd.DataFrame(rows).set_index("model"); print(t.round(4).to_string())
assert abs(t.iloc[0, 0] - 0.0684) < 0.0005 and abs(t.iloc[1, 0] - 0.0950) < 0.0005, "does not reproduce E24"
b, n = t.iloc[0], t.iloc[2]; gain = n["D2 all perils"] - b["D2 all perils"]
tails = all(abs(n[kk] - 1) <= abs(b[kk] - 1) + 0.1 for kk in ["low, largest removed", "top, largest removed"])
print(f"\nE25: D2 {gain:+.4f} | tails (largest removed) {'ok' if tails else 'FAIL'} -> {'ADOPT' if gain >= 0.005 and tails else 'not adopted'}")

21:07:39 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:07:40 buildings: fire priced separately at 97.05 per exposure (developed)
21:07:40 buildings: own GLM for ['flood']; main GLM on the other perils
CV D2 gain inside 2023 (top 12):
wind_exposure_index    0.0003
wildfire_index         0.0002
smoke_alarms           0.0002
new_business           0.0002
energy_rating          0.0002
roof_type              0.0001
property_type          0.0001
solar_panels           0.0001
years_at_address       0.0001
hail_days              0.0001
dry_summer_index       0.0001
voluntary_excess       0.0000
selected: []
                              D2 all perils  low (rule b)  top (rule b)  low, largest removed  top, largest removed
model                                                                                                              
current (emerging only)              0.0684        0.6154        1.0215                0.5149                0.8889
combined, dev w

In [13]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "  neither tail moves more than 0.1 further from 1 than the current model. 2024 is used once.\n"
assert d.count(a) == 1 and "**E25 result:**" not in d
d = d.replace(a, a + "- **E25 result:** no feature passed the screen inside 2023 (largest CV gain +0.0003: wind_exposure_index; then wildfire_index, "
              "smoke_alarms, new_business, energy_rating +0.0002), so E25 = E24 (combined): 2024 D2 +0.0266, lowest decile (largest removed) 0.515 -> 0.320. "
              "**Not adopted.** Buildings other perils stay on emerging data only.\n"
              "- Reading: the combined model's over-pricing of the safest 2024 decile is not visible in 2023 as a feature-level difference "
              "(the year, an overall shape, or chance). A Buhlmann-Straub correction (E26) was considered and not run: with between-level variance "
              "about zero in 2023 its credibility would be near 0 and reproduce E24, at the cost of a third use of 2024.\n"
              "- For the report: protecting the safest decile costs about +0.027 buildings validation D2; the attempt to keep the gain found no basis in 2023.\n")
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [14]:
import os, py_compile; from pathlib import Path; os.chdir(Path.home()/"IAA")
f = Path("src/target.py"); t = f.read_text(encoding="utf-8")
assert "excess_in_amount" not in t, "already patched"
o1 = ('    if AMOUNTS_NET if amounts_net is None else amounts_net:\n'
      '        c["excess"] = 0.0   # amounts are already net of excess: do not deduct it again\n')
n1 = ('    c["excess_in_amount"] = 0.0\n'
      '    if AMOUNTS_NET if amounts_net is None else amounts_net:\n'
      '        c["excess_in_amount"] = c["excess"]   # already deducted in the amounts; kept to trend the ground-up loss (Oct 24)\n'
      '        c["excess"] = 0.0   # amounts are already net of excess: do not deduct it again\n')
o2 = ('    gross = c["incurred_to_date"] * sev\n'
      '    rec = c["recoveries"] * sev\n'
      '    net = np.maximum(np.maximum(gross - c["excess"], 0) - rec, 0)\n')
n2 = ('    add = c["excess_in_amount"] if "excess_in_amount" in c else 0.0   # Oct 24: net amounts -> trend ground-up, excess fixed (leverage)\n'
      '    gross = (c["incurred_to_date"] + add) * sev\n'
      '    rec = c["recoveries"] * sev\n'
      '    net = np.maximum(np.maximum(gross - c["excess"] - add, 0) - rec, 0)\n')
assert t.count(o1) == 1 and t.count(o2) == 1, "target.py differs from the expected lines"
f.write_text(t.replace(o1, n1).replace(o2, n2), encoding="utf-8"); py_compile.compile(str(f), doraise=True)

dec = """### Trend leverage restored for net amounts (Oct 24, notebook 28) — definition, not score
- Register (ASSUMPTIONS 4): trends apply to the ground-up loss with the excess fixed, so the net grows faster (leverage).
  With AMOUNTS_NET the excess was set to 0 and the trend multiplied the net amount: leverage lost.
- Fix (src/target.py): match_claims keeps the excess already deducted in the amounts (excess_in_amount); adjust_claims trends
  incurred + excess and deducts the excess again. Untrended amounts are unchanged; tied to AMOUNTS_NET (every entry point, developed table included).
- Effect measured on emerging training claims (trended totals): motor x1.017, buildings x1.006, contents x1.010.
  Frequency leverage (losses newly above the excess) is still not modelled; it would raise the level further.
- **p2_v7 (v6 + leverage) expected public, written before submitting: 0.1426-0.1440.** Adopted by definition; v7 becomes the final
  unless its public score is below 0.1420 (then look for a bug). New-business load unchanged (x1.3).
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "Trend leverage restored" not in d
p.write_text(d.replace(a, dec + "\n" + a), encoding="utf-8")

log = ("| 112 | E | When the Oct 24 excess fix set the excess to 0 for net amounts, AI did not carry the register's trend rule (ground-up, excess fixed), "
       "so the severity trend multiplied the net amount and the leverage was lost. "
       "| Found the same evening while reviewing time conventions (exact-date trending, seasonality) at the team's request. "
       "| Fixed in target.py, tied to AMOUNTS_NET; level x1.017 / 1.006 / 1.010. Lesson: a change to the target definition is checked against every register line that uses it. |\n")
q = Path("AI_USE_LOG.md"); lines = q.read_text(encoding="utf-8").splitlines(keepends=True)
assert not any(l.startswith("| 112 |") for l in lines)
last = max(i for i, l in enumerate(lines) if l.startswith("| 11"))
lines.insert(last + 1, log); q.write_text("".join(lines), encoding="utf-8")
print("patched, recorded")

patched, recorded


In [15]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "  unless its public score is below 0.1420 (then look for a bug). New-business load unchanged (x1.3).\n"
assert d.count(a) == 1 and "p2_v7 public" not in d
d = d.replace(a, a + "- Run: totals v7/v6 motor +1.17%, buildings +0.73%, contents +0.83% (all +1.03%), correlation >= 0.9999; tests passed.\n"
              "- **p2_v7 public (Oct 24, 21:12): 0.14336**, inside 0.1426-0.1440; +0.0001 over v6. **p2_v7_leverage is the Part 2 final.**\n")
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [16]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### H2a: do new-business relativities differ from renewals? (Oct 24, notebook 28) — rule fixed before running
- Question behind the broker-book ranking hypothesis, tested where new business is plentiful: the developed market (2021-24). No emerging 2024 use.
- Base o: GLM on developed 2021-23, all rows (Part 1 GLM settings, the features shared with emerging), per line.
- For each feature: a one-feature correction on top of o (offset via y/o, weight w*sqrt(o); alpha 10), fitted on 2021-23 new-business rows,
  scored on 2024 new-business rows (out of time). Control: the same on renewal rows. Signal = new-business gain - renewal gain.
- Rule: any feature with signal >= +0.002 (any line) -> new-business relativities differ; candidate = new-business corrections for those
  features, tested once on the emerging 2024 fold under the GLM rule. All signals < +0.001 -> no support; the broker-book tilt probes (2b) are dropped.
  In between: recorded only.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
assert "### H2a:" not in d, "already recorded"
a = "## Assumptions register"; assert d.count(a) == 1
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [17]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])
def d2(a, pv, w): return evaluate(pd.Series(np.asarray(a)), np.asarray(pv), pd.Series(np.asarray(w)), show=False)["D2"]
def corr_fit(X, r, w, o, cols):
    return BandedTweedieGLM(power=1.5, alpha=10, n_bands=10, drop=None, balance=True).fit(X[cols], r / o, w * np.sqrt(o))

def gain(fit, test, o_fit, o_test, col):
    k0 = fit.target.sum() / (o_fit * fit.weight).sum()
    c = corr_fit(fit, (fit.target / fit.weight).to_numpy(), fit.weight.to_numpy(), o_fit, [col])
    k1 = fit.target.sum() / (o_fit * c.predict_rate(fit[[col]]) * fit.weight).sum()
    return (d2(test.target, o_test * c.predict_rate(test[[col]]) * k1 * test.weight, test.weight)
            - d2(test.target, o_test * k0 * test.weight, test.weight))

out = []
for line in cfg["lines"]:
    feats = P.build_line(line, cfg, claims)["feats"]
    dev = P.developed_table(line, cfg).reset_index(drop=True)
    f = [c for c in feats if c in dev.columns]
    old = dev.year <= 2023
    g = BandedTweedieGLM(**PART1["glm"][line]).fit(dev.loc[old, f], dev.target[old] / dev.weight[old], dev.weight[old])
    o = np.asarray(g.predict_rate(dev[f]))
    nb = dev.new_business.astype(bool).to_numpy()
    P.log(f"{line}: developed rows {len(dev):,}, new-business share {nb.mean():.1%} "
          f"(2021-23 {(nb & old).sum():,}, 2024 {(nb & ~old).sum():,})")
    cand = [c for c in f if c not in ("new_business", "renewal_count") and dev[c].nunique() > 1]
    for c in cand:
        row = {"line": line, "feature": c}
        for name, m in [("new business", nb), ("renewal", ~nb)]:
            fi, te = (m & old.to_numpy()), (m & ~old.to_numpy())
            row[f"gain {name}"] = gain(dev[fi], dev[te], o[fi], o[te], c)
        row["signal"] = row["gain new business"] - row["gain renewal"]
        out.append(row)
r = pd.DataFrame(out)
for line, gdf in r.groupby("line", sort=False):
    print(f"\n== {line} (top 10 by signal) ==")
    print(gdf.sort_values("signal", ascending=False).head(10).round(4).to_string(index=False))
mx = r.signal.max()
print(f"\nlargest signal {mx:+.4f} -> " + ("NEW-BUSINESS RELATIVITIES DIFFER (>= 0.002)" if mx >= 0.002
      else "no support (< 0.001): drop 2b" if mx < 0.001 else "in between: record only"))
r.to_csv("results/h2a_nb_relativities.csv", index=False)

21:13:59 motor: developed rows 979,968, new-business share 14.5% (2021-23 103,419, 2024 39,103)
21:14:28 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:14:28 buildings: fire priced separately at 97.05 per exposure (developed)
21:14:28 buildings: own GLM for ['flood']; main GLM on the other perils
21:14:47 buildings: developed rows 707,726, new-business share 14.6% (2021-23 75,134, 2024 28,030)
21:15:09 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
21:15:12 contents: developed training table 721,617 rows
21:15:34 contents: developed rows 721,617, new-business share 14.6% (2021-23 76,670, 2024 28,540)

== motor (top 10 by signal) ==
 line          feature  gain new business  gain renewal  signal
motor   licence_points                0.0          -0.0     0.0
motor      vehicle_use                0.0           0.0     0.0
motor  prior_claims_5y                0.0           0.0     0.0
motor voluntary_excess                0.0         

In [18]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### Ridge scale error in offset corrections (E21, E25, H2a) — found Oct 24, 21:17 (notebook 28) [AI 113]
- Corrections were fitted on y/o with weight w*sqrt(o). sklearn's penalty is on the weighted mean deviance, and Tweedie p=1.5 deviance scales
  with sqrt(amount), so alpha on this scale equals alpha on the cost scale times mean(sqrt(o)) (motor about 22, buildings about 17, contents about 9.5).
- Check (emerging motor 2023, one feature): driver_age_band spread at alpha 10 is 1.37 on the cost scale but 1.02 on the ratio scale.
- Affected: E21 (alpha 10 / 2), E25 (alpha 10), H2a (alpha 10): corrections were shrunk almost to zero (H2a's renewal control was exactly 0).
  Their conclusions are withdrawn. Fix: alpha_ratio = alpha_cost / weighted mean of sqrt(o). Same rules re-run; E25 uses 2024 a second time (disclosed).
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "Ridge scale error in offset corrections" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8")
log = ("| 113 | E | AI fitted offset corrections on y/o with weight w*sqrt(o) and kept the cost-scale ridge alpha; on that scale the penalty is about "
       "mean(sqrt(o)) times stronger (motor about 22x), so E21, E25 and H2a shrank the corrections almost to zero and reported 'no gain'. "
       "| Found because H2a's renewal control was exactly 0 for every feature; confirmed by fitting one feature on both scales (spread 1.37 vs 1.02). "
       "| Conclusions withdrawn; alpha rescaled; same rules re-run. Lesson: a null result with a null control is checked for a dead instrument before it is read. |\n")
q = Path("AI_USE_LOG.md"); lines = q.read_text(encoding="utf-8").splitlines(keepends=True)
assert not any(l.startswith("| 113 |") for l in lines)
last = max(i for i, l in enumerate(lines) if l.startswith("| 11"))
lines.insert(last + 1, log); q.write_text("".join(lines), encoding="utf-8"); print("recorded")

recorded


In [20]:
A_COST = 0.5   # Part 1 developed alpha, cost scale (fixed before the re-run)
def corr_fit(X, r, w, o, cols, a_cost=A_COST):
    a = a_cost / np.average(np.sqrt(o), weights=w)          # cost-scale alpha -> ratio scale
    return BandedTweedieGLM(power=1.5, alpha=a, n_bands=10, drop=None, balance=True).fit(X[cols], r / o, w * np.sqrt(o))

def gain(fit, test, o_fit, o_test, col):
    k0 = fit.target.sum() / (o_fit * fit.weight).sum()
    c = corr_fit(fit, (fit.target / fit.weight).to_numpy(), fit.weight.to_numpy(), o_fit, [col])
    k1 = fit.target.sum() / (o_fit * c.predict_rate(fit[[col]]) * fit.weight).sum()
    return (d2(test.target, o_test * c.predict_rate(test[[col]]) * k1 * test.weight, test.weight)
            - d2(test.target, o_test * k0 * test.weight, test.weight))

In [21]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### Developed vs emerging, peril level, and the borrowed amount assumptions (Oct 24, 21:20, notebook 28)
- 2023 policy year (mature in both), net of excess and recoveries, no trend, uncapped; ratio emerging / developed:
  motor frequency x1.35, severity x0.55, cost x0.74 (BI severity x0.45, glass frequency x1.55); buildings x1.00 / x1.18 / x1.19
  (fire x3.0 = 2023 cluster; escape of water x0.45); contents x0.85 / x0.92 / x0.78. Emerging: more claims, cheaper claims (motor).
  Largest mix difference: channel (TVD 0.45). construction_type TVD 1.00 is the raw local codes (mapped in the pipeline).
- Checked because severity levels differ:
  1. Motor Pareto (borrowed alpha 2.05, threshold 100k): emerging has 3 claims > 100k, 8 > 55k; loading 0.02%-0.49% of the rate
     under any threshold / alpha (current 0.31%). No change.
  2. Caps: no emerging motor claim > 500k; contents uncapped; buildings 300k cuts one claim (-1.0%). No change.
  3. Severity trends (not testable; sensitivity for the report and the office premium table): all trends -2 / +2 points:
     motor -2.4% / +3.5%, buildings -1.8% / +3.0%, contents -3.4% / +3.4%.
  4. Fire: emerging 2024 severity 53,585 (median 28,466) and frequency 1.74 per 1,000 match developed 2021-24
     (50,509-54,347; 27,240-29,850; 1.68-2.06). The x1.3 severity is the 2023 cluster. Developed rate 97.05 kept.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "Developed vs emerging, peril level" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [23]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])
def d2(a, pv, w): return evaluate(pd.Series(np.asarray(a)), np.asarray(pv), pd.Series(np.asarray(w)), show=False)["D2"]

A_COST = 0.5   # Part 1 developed alpha, cost scale (fixed before the re-run)
def corr_fit(X, r, w, o, cols, a_cost=A_COST):
    a = a_cost / np.average(np.sqrt(o), weights=w)          # cost-scale alpha -> ratio scale
    return BandedTweedieGLM(power=1.5, alpha=a, n_bands=10, drop=None, balance=True).fit(X[cols], r / o, w * np.sqrt(o))

def gain(fit, test, o_fit, o_test, col):
    k0 = fit.target.sum() / (o_fit * fit.weight).sum()
    c = corr_fit(fit, (fit.target / fit.weight).to_numpy(), fit.weight.to_numpy(), o_fit, [col])
    k1 = fit.target.sum() / (o_fit * c.predict_rate(fit[[col]]) * fit.weight).sum()
    return (d2(test.target, o_test * c.predict_rate(test[[col]]) * k1 * test.weight, test.weight)
            - d2(test.target, o_test * k0 * test.weight, test.weight))

out = []
for line in cfg["lines"]:
    feats = P.build_line(line, cfg, claims)["feats"]
    dev = P.developed_table(line, cfg).reset_index(drop=True)
    f = [c for c in feats if c in dev.columns]
    old = (dev.year <= 2023).to_numpy()
    g = BandedTweedieGLM(**PART1["glm"][line]).fit(dev.loc[old, f], dev.target[old] / dev.weight[old], dev.weight[old])
    o = np.asarray(g.predict_rate(dev[f]))
    nb = dev.new_business.astype(bool).to_numpy()
    P.log(f"{line}: developed rows {len(dev):,}, new business 2021-23 {(nb & old).sum():,}, 2024 {(nb & ~old).sum():,}")
    for c in [c for c in f if c not in ("new_business", "renewal_count") and dev[c].nunique() > 1]:
        row = {"line": line, "feature": c}
        for name, m in [("new business", nb), ("renewal", ~nb)]:
            fi, te = m & old, m & ~old
            row[f"gain {name}"] = gain(dev[fi], dev[te], o[fi], o[te], c)
        row["signal"] = row["gain new business"] - row["gain renewal"]
        out.append(row)
r = pd.DataFrame(out)
for line, gdf in r.groupby("line", sort=False):
    print(f"\n== {line} (top 10 by signal) ==")
    print(gdf.sort_values("signal", ascending=False).head(10).round(4).to_string(index=False))
print(f"\ncontrol check: renewal gains range {r['gain renewal'].min():+.4f} to {r['gain renewal'].max():+.4f} (should not be all zero)")
mx = r.signal.max()
print(f"largest signal {mx:+.4f} -> " + ("NEW-BUSINESS RELATIVITIES DIFFER (>= 0.002)" if mx >= 0.002
      else "no support (< 0.001): drop 2b" if mx < 0.001 else "in between: record only"))
r.to_csv("results/h2a_nb_relativities_rerun.csv", index=False)

21:21:07 motor: developed rows 979,968, new business 2021-23 103,419, 2024 39,103
21:21:35 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:21:35 buildings: fire priced separately at 97.05 per exposure (developed)
21:21:36 buildings: own GLM for ['flood']; main GLM on the other perils
21:21:55 buildings: developed rows 707,726, new business 2021-23 75,134, 2024 28,030
21:22:17 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
21:22:38 contents: developed rows 721,617, new business 2021-23 76,670, 2024 28,540

== motor (top 10 by signal) ==
 line          feature  gain new business  gain renewal  signal
motor  listed_building             0.0008       -0.0000  0.0008
motor   licence_points             0.0004       -0.0001  0.0005
motor  trees_within_5m             0.0004       -0.0000  0.0005
motor  driver_age_band             0.0002       -0.0001  0.0004
motor   years_licensed             0.0003       -0.0000  0.0004
motor      rainfall_m

In [24]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "  In between: recorded only.\n"
assert d.count(a) == 1 and "**H2a result (re-run" not in d
d = d.replace(a, a + "- **H2a result (re-run after the ridge-scale fix, AI 113; cost-scale alpha 0.5):** renewal control -0.0004 to +0.0007 (instrument alive). "
              "Motor: no feature >= 0.002 (largest listed_building +0.0008): new-business relativities match renewals. "
              "Buildings: rainfall_mm +0.0057, flood_zone +0.0026. Contents: distance_to_river_km +0.0027, freezing_days +0.0024, energy_rating +0.0023, "
              "building_age +0.0021, voluntary_excess +0.0020. **Rule met (home lines): new-business relativities differ, mainly location and weather.**\n"
              "- Caveats: about 120 feature tests (some may pass by chance; new-business 2024 sample about 28k rows per line); developed new business "
              "is not the broker book (indirect evidence).\n"
              "- Next (E26, to be fixed before running): new-business corrections for these features, fitted on developed new business 2021-24 "
              "(offset on the developed base), applied to emerging new-business rows only with each line's new-business total held fixed. "
              "The emerging 2024 fold has too few new-business losses to judge (buildings 35, contents 64); adoption rule to be decided by the team.\n")
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [25]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### E26: new-business corrections for home lines from developed new business (Oct 24, notebook 28) — rule fixed before running (decided by Woojin)
- Features (passed H2a): buildings rainfall_mm, flood_zone; contents distance_to_river_km, freezing_days, energy_rating, building_age,
  voluntary_excess. Motor unchanged (H2a: no difference).
- Fit: developed base GLM on all developed rows 2021-24 (Part 1 settings); one correction GLM per line on developed new-business rows
  2021-24 (offset on the base; cost-scale alpha 0.5 converted to the ratio scale, AI 113).
- Apply: emerging new-business rows only; each line's new-business total held equal to v7 (the x1.3 load unchanged; ranking only);
  correction factors clipped to 0.67-1.5 (bands come from developed weather and location). Renewals unchanged.
- Emerging 2024 fold: direction check only (new-business losses: buildings 35, contents 64), not a decision input.
- Adoption (option a, team): adopted on the developed out-of-time evidence (H2a). The leaderboard is a guard only:
  p2_v8 is withdrawn (v7 stays final) if its public score is below v7 - 0.001 = 0.14236. **Expected public: 0.1433-0.1455.**
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "### E26:" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [27]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])
SEL = {"buildings": ["rainfall_mm", "flood_zone"],
       "contents": ["distance_to_river_km", "freezing_days", "energy_rating", "building_age", "voluntary_excess"]}
A_COST, CLIP = 0.5, (0.67, 1.5)
def d2(a, pv, w): return evaluate(pd.Series(np.asarray(a)), np.asarray(pv), pd.Series(np.asarray(w)), show=False)["D2"]
def corr_fit(X, r, w, o, cols):
    a = A_COST / np.average(np.sqrt(o), weights=w)
    return BandedTweedieGLM(power=1.5, alpha=a, n_bands=10, drop=None, balance=True).fit(X[cols], r / o, w * np.sqrt(o))

v7 = pd.read_csv("submissions/p2_v7_leverage.csv"); idc, valc = v7.columns[:2]
pred = v7.set_index(idc)[valc].copy()
for line, sel in SEL.items():
    d = P.build_line(line, cfg, claims)
    dev = P.developed_table(line, cfg).reset_index(drop=True)
    f = [c for c in d["feats"] if c in dev.columns]; assert all(c in f for c in sel)
    g = BandedTweedieGLM(**PART1["glm"][line]).fit(dev[f], dev.target / dev.weight, dev.weight)
    o = np.asarray(g.predict_rate(dev[f])); nb = dev.new_business.astype(bool).to_numpy(); dn = dev[nb]
    corr = corr_fit(dn, (dn.target / dn.weight).to_numpy(), dn.weight.to_numpy(), o[nb], sel)

    te = d["te"]; tnb = te.new_business.astype(bool).to_numpy(); ids = te.row_id[tnb].to_numpy()
    raw = np.asarray(corr.predict_rate(te.loc[tnb, sel])); fac = np.clip(raw, *CLIP)
    old = pred.loc[ids].to_numpy(); new = old * fac * old.sum() / (old * fac).sum()
    pred.loc[ids] = new
    q = np.percentile(fac, [1, 10, 50, 90, 99])
    P.log(f"{line}: test new-business rows {tnb.sum():,}; factor p1/p10/p50/p90/p99 " + " / ".join(f"{x:.3f}" for x in q)
          + f"; clipped {np.mean((raw < CLIP[0]) | (raw > CLIP[1])):.2%}")

    tr, va = d["tr"][d["tr"].year == 2023], d["tr"][d["tr"].year == 2024]
    rate, _ = P.predict_rate(line, cfg, tr, va, d["feats"]); w = va.weight.to_numpy()
    pv0 = (np.asarray(rate) + d["sep_rate"]) * w; vnb = va.new_business.astype(bool).to_numpy()
    pv1 = pv0.copy(); pv1[vnb] = pv0[vnb] * np.clip(np.asarray(corr.predict_rate(va.loc[vnb, sel])), *CLIP)
    pv1[vnb] *= pv0[vnb].sum() / pv1[vnb].sum()
    act = (va.target_all if "target_all" in va else va.target).to_numpy()
    print(f"  {line} emerging 2024 (direction only): D2 all rows {d2(act, pv0, w):.4f} -> {d2(act, pv1, w):.4f}; "
          f"new-business rows {d2(act[vnb], pv0[vnb], w[vnb]):.4f} -> {d2(act[vnb], pv1[vnb], w[vnb]):.4f}")

v8 = pred.reset_index(); v8.columns = [idc, valc]
assert list(v8[idc]) == list(v7[idc]) and v8[valc].notna().all() and (v8[valc] >= 0).all()
L = v8[idc].str[0]
print("\ntotals v8/v7 by line:", (v8[valc].groupby(L).sum() / v7[valc].groupby(L).sum()).round(6).to_dict())
assert np.allclose(v8[valc][L == "M"], v7[valc][L == "M"]), "motor must be unchanged"
print("rows changed:", f"{(~np.isclose(v8[valc], v7[valc])).mean():.1%}",
      "| correlation with v7:", round(np.corrcoef(v8[valc], v7[valc])[0, 1], 5))
v8.to_csv("submissions/p2_v8_nbcorr.csv", index=False); print("saved submissions/p2_v8_nbcorr.csv")

21:29:59 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:30:00 buildings: fire priced separately at 97.05 per exposure (developed)
21:30:00 buildings: own GLM for ['flood']; main GLM on the other perils
21:30:24 buildings: test new-business rows 39,716; factor p1/p10/p50/p90/p99 0.670 / 0.725 / 0.865 / 1.021 / 1.036; clipped 8.73%
  buildings emerging 2024 (direction only): D2 all rows 0.0684 -> 0.0690; new-business rows -0.0220 -> -0.0158
21:30:32 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
21:31:00 contents: test new-business rows 50,441; factor p1/p10/p50/p90/p99 0.670 / 0.720 / 0.907 / 1.122 / 1.307; clipped 4.77%
  contents emerging 2024 (direction only): D2 all rows 0.0553 -> 0.0552; new-business rows 0.0222 -> 0.0212

totals v8/v7 by line: {'B': 1.0, 'C': 1.0, 'M': 1.0}
rows changed: 43.1% | correlation with v7: 0.9947
saved submissions/p2_v8_nbcorr.csv


In [28]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### Diagnostic: actual 2025 weather (Oct 24, notebook 28) — written before submitting (decided by Woojin)
- p2_diag_weather_actual = v7 with weather_mode "actual" (training rows get their own year's weather, test rows 2025 weather); nothing else changed.
- Why: 2025 weather is unusual (freezing days 17 -> 29, hail days 17 -> 11); E19 found no gain in the 2024 fold (a normal year).
- Expected vs v7 (0.14336): >= +0.005 -> 2025 weather explains a large part of the gap to the leaders; within +-0.002 -> not the cause.
- **Diagnostic only: the final selection does not use 2025 weather** (unknown when pricing; Part 1 E4, E19). Reported as the gap that
  information unavailable at pricing time would close.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "Diagnostic: actual 2025 weather" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded


In [29]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P
from src.config import PART2

cfg = copy.deepcopy(PART2); cfg["weather_mode"] = "actual"
out = "submissions/p2_diag_weather_actual.csv"
P.run(cfg, out)

a = pd.read_csv("submissions/p2_v7_leverage.csv"); b = pd.read_csv(out)
assert list(a.row_id) == list(b.row_id) and b.prediction.notna().all() and (b.prediction >= 0).all()
L = a.row_id.str[0]
print(pd.DataFrame({"total diag/v7": b.prediction.groupby(L).sum() / a.prediction.groupby(L).sum(),
                    "correlation": pd.Series({k: np.corrcoef(a.prediction[L == k], b.prediction[L == k])[0, 1] for k in "BCM"})}).round(4))

21:31:55 motor: GLM on developed data (42 features), emerging level x0.7902; Pareto, alpha 2.05 borrowed: 3 claims above 100,000, loading 1.52 -> factor 1.0031; broker-book load x1.3 on 76% of rows; mean rate 588.61; total 58,071,006; completeness {2023: 0.982, 2024: 0.804} (26s)
21:31:55 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:31:55 buildings: fire priced separately at 97.05 per exposure (developed)
21:31:55 buildings: own GLM for ['flood']; main GLM on the other perils
21:32:04 buildings: GLM, k=1.0594 + flood GLM on combined data (17 features, alpha 0.5, level x0.7920); no loading; + separate peril rate 97.05; broker-book load x1.3 on 76% of rows; mean rate 355.17; total 20,728,449; completeness {2023: 1.0, 2024: 0.95} (10s)
21:32:04 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
21:32:27 contents: GLM on combined data (46 features), emerging level x0.9640; no loading; broker-book load x1.3 on 76% of rows; mean rate 98.38;

In [30]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "  p2_v8 is withdrawn (v7 stays final) if its public score is below v7 - 0.001 = 0.14236. **Expected public: 0.1433-0.1455.**\n"
assert d.count(a) == 1 and "p2_v8 public" not in d
d = d.replace(a, a + "- Run: test new-business factors (after clipping) buildings p10/p50/p90 0.725 / 0.865 / 1.021, clipped 8.7%; contents "
              "0.720 / 0.907 / 1.122, clipped 4.8%; totals by line equal to v7; correlation with v7 0.9947. Emerging 2024 (direction only): "
              "buildings new-business D2 -0.0220 -> -0.0158, contents 0.0222 -> 0.0212.\n"
              "- **p2_v8 public (Oct 24, 21:35): 0.14077**, below the guard 0.14236 (-0.0026 vs v7). **Withdrawn; v7 stays final.** "
              "Developed new-business relativities do not carry to the broker book, or the developed weather bands do not transfer "
              "(buildings factors skewed, 8.7% clipped). The guard worked as intended.\n")
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [31]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"])
A_COST = {"motor": 10, "contents": 2}           # E21's CV values, cost scale (unchanged)
def metrics(va, rate):
    r = evaluate(va.target, np.asarray(rate) * va.weight.to_numpy(), va.weight, show=False); lf = r["lift"]["A/E"]
    return {"D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1]}

rows = []
for line in ["motor", "contents"]:
    d = P.build_line(line, cfg, claims); feats = d["feats"]
    tr, va = d["tr"][d["tr"].year == 2023], d["tr"][d["tr"].year == 2024]
    cur, _ = P.predict_rate(line, cfg, tr, va, feats)                     # current model (v7 structure)
    dev = P.developed_table(line, cfg); f = [c for c in feats if c in dev.columns]
    g = BandedTweedieGLM(**PART1["glm"][line]).fit(dev[f], dev.target / dev.weight, dev.weight)
    o_tr, o_va = np.asarray(g.predict_rate(tr[f])), np.asarray(g.predict_rate(va[f]))
    r_tr, w_tr = (tr.target / tr.weight).to_numpy(), tr.weight.to_numpy()
    rows.append({"line": line, "model": "current (v7)", **metrics(va, cur)})
    for mult in [1.0, 0.3, 3.0]:
        a = A_COST[line] * mult / np.average(np.sqrt(o_tr), weights=w_tr)   # cost scale -> ratio scale (AI 113)
        c = BandedTweedieGLM(power=1.5, alpha=a, n_bands=cfg["glm"][line]["n_bands"], drop=None, balance=True) \
            .fit(tr[feats], r_tr / o_tr, w_tr * np.sqrt(o_tr))
        k = tr.target.sum() / (o_tr * c.predict_rate(tr[feats]) * w_tr).sum()
        rows.append({"line": line, "model": f"E21 offset + emerging, alpha x{mult}", **metrics(va, o_va * c.predict_rate(va[feats]) * k)})
    P.log(f"{line} done")

t = pd.DataFrame(rows); print(t.round(4).to_string(index=False))
print("\nRule (alpha x1 only): D2 +0.0005 and tails no more than 0.1 further from 1")
for line in ["motor", "contents"]:
    b = t[(t.line == line) & (t.model == "current (v7)")].iloc[0]; n = t[(t.line == line) & (t.model.str.endswith("x1.0"))].iloc[0]
    gain = n.D2 - b.D2; tails = all(abs(n[k] - 1) <= abs(b[k] - 1) + 0.1 for k in ["low", "top"])
    print(f"  {line}: D2 {gain:+.4f} | tails {'ok' if tails else 'FAIL'} -> {'ADOPT' if gain >= 0.0005 and tails else 'not adopted'}")

21:37:59 motor done
21:37:59 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
21:38:44 contents done
    line                             model     D2    A/E    low    top
   motor                      current (v7) 0.0373 1.0764 0.7940 0.8936
   motor E21 offset + emerging, alpha x1.0 0.0360 1.0674 0.8474 0.8367
   motor E21 offset + emerging, alpha x0.3 0.0285 1.0614 1.0809 0.7408
   motor E21 offset + emerging, alpha x3.0 0.0378 1.0721 0.7664 0.8821
contents                      current (v7) 0.0553 0.9256 0.8555 0.9723
contents E21 offset + emerging, alpha x1.0 0.0411 0.9252 0.9739 0.6461
contents E21 offset + emerging, alpha x0.3 0.0005 0.9203 1.4256 0.4250
contents E21 offset + emerging, alpha x3.0 0.0537 0.9269 0.8082 0.9307

Rule (alpha x1 only): D2 +0.0005 and tails no more than 0.1 further from 1
  motor: D2 -0.0014 | tails ok -> not adopted
  contents: D2 -0.0142 | tails FAIL -> not adopted


In [32]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "  information unavailable at pricing time would close.\n"
assert d.count(a) == 1 and "p2_diag_weather_actual public" not in d
d = d.replace(a, a + "- Run: totals vs v7 motor 0.9998, buildings 0.9791, contents 0.9756; correlation 0.988-0.996.\n"
              "- **p2_diag_weather_actual public (Oct 24, 21:37): 0.14399**, +0.0006 vs v7: within +-0.002, so 2025 weather is not the cause "
              "of the gap to the leaders. Not used for the final (v7 stays). Value of information unavailable at pricing time: about +0.0006 public.\n")
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [33]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "  Their conclusions are withdrawn. Fix: alpha_ratio = alpha_cost / weighted mean of sqrt(o). Same rules re-run; E25 uses 2024 a second time (disclosed).\n"
assert d.count(a) == 1 and "**E21 re-run:**" not in d
d = d.replace(a, a + "- **E21 re-run:** (cost-scale alpha motor 10, contents 2, rescaled) motor D2 0.0373 -> 0.0360 (-0.0014), contents 0.0553 -> 0.0411 "
              "(-0.0142, top decile 0.97 -> 0.65): **not adopted.** Sensitivity: alpha x0.3 motor 0.0285, contents 0.0005; x3 motor 0.0378, contents 0.0537. "
              "The less shrinkage, the worse: one year of emerging data over-fits emerging-specific relativities. Market levels differ "
              "(motor frequency x1.35, severity x0.55) but the developed relativities transfer.\n")
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [34]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"]); line = "buildings"
d = P.build_line(line, cfg, claims); feats = d["feats"]
tr0 = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id")
tr = tr0[tr0.year == 2023].reset_index(drop=True)
dev = P.developed_table(line, cfg); f = [c for c in feats if c in dev.columns]
both = pd.concat([dev[f + ["target", "weight"]].assign(market="developed"),
                  tr[f + ["target", "weight"]].assign(market="emerging")], ignore_index=True)
g = BandedTweedieGLM(**PART1["glm"][line]).fit(both[f + ["market"]], both.target / both.weight, both.weight)
o_tr = np.asarray(g.predict_rate(tr[f].assign(market="emerging"))); o_tr *= tr.target.sum() / (o_tr * tr.weight).sum()
r_tr, w_tr = (tr.target / tr.weight).to_numpy(), tr.weight.to_numpy()
a = 10 / np.average(np.sqrt(o_tr), weights=w_tr)                      # cost-scale 10 -> ratio scale (AI 113)
def d2(act, pv, w): return evaluate(pd.Series(np.asarray(act)), np.asarray(pv), pd.Series(np.asarray(w)), show=False)["D2"]
grp = (pd.util.hash_pandas_object(tr.policy_id.astype(str), index=False) % 5).to_numpy()
def cv_gain(cols):
    bp, npv = np.zeros(len(tr)), np.zeros(len(tr))
    for k in range(5):
        fi, ho = grp != k, grp == k
        k0 = tr.target[fi].sum() / (o_tr[fi] * w_tr[fi]).sum(); bp[ho] = o_tr[ho] * k0 * w_tr[ho]
        c = BandedTweedieGLM(power=1.5, alpha=a, n_bands=10, drop=None, balance=True).fit(tr[fi][cols], r_tr[fi] / o_tr[fi], w_tr[fi] * np.sqrt(o_tr[fi]))
        k1 = tr.target[fi].sum() / (o_tr[fi] * c.predict_rate(tr[fi][cols]) * w_tr[fi]).sum()
        npv[ho] = o_tr[ho] * c.predict_rate(tr[ho][cols]) * k1 * w_tr[ho]
    return d2(tr.target, npv, w_tr) - d2(tr.target, bp, w_tr)
scr = pd.Series({c: cv_gain([c]) for c in feats}).sort_values(ascending=False)
print("CV D2 gain inside 2023 (top 12):"); print(scr.head(12).round(4).to_string())
sel = [c for c in scr.index if scr[c] >= 0.001][:5]
print("\nselected (>= +0.001):", sel if sel else "none -> E25 stands as no-feature; 2024 not used")

21:40:27 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:40:27 buildings: fire priced separately at 97.05 per exposure (developed)
21:40:27 buildings: own GLM for ['flood']; main GLM on the other perils
CV D2 gain inside 2023 (top 12):
wind_exposure_index    0.0028
wildfire_index         0.0023
new_business           0.0021
energy_rating          0.0020
smoke_alarms           0.0017
roof_type              0.0010
solar_panels           0.0009
property_type          0.0008
years_at_address       0.0008
dry_summer_index       0.0007
hail_days              0.0006
freezing_days          0.0003

selected (>= +0.001): ['wind_exposure_index', 'wildfire_index', 'new_business', 'energy_rating', 'smoke_alarms']


In [35]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "- For the report: protecting the safest decile costs about +0.027 buildings validation D2; the attempt to keep the gain found no basis in 2023.\n"
assert d.count(a) == 1 and "**E25 re-run (screen" not in d
d = d.replace(a, a + "- **E25 re-run (screen inside 2023 only, ridge-scale fixed, AI 113; cost-scale alpha 10):** selected wind_exposure_index +0.0028, "
              "wildfire_index +0.0023, new_business +0.0021, energy_rating +0.0020, smoke_alarms +0.0017 (CV D2 gains). The earlier 'no basis in 2023' "
              "is withdrawn. These five are now judged once on 2024 under the unchanged E25 rule; this is the third use of 2024 for the buildings "
              "combined question (E24, E25 first run, this run), disclosed before running.\n")
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [36]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART1, PART2
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate

SEL = ["wind_exposure_index", "wildfire_index", "new_business", "energy_rating", "smoke_alarms"]   # fixed by the 2023 screen
cfg = copy.deepcopy(PART2); T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(cfg["data_dir"]); line = "buildings"
d = P.build_line(line, cfg, claims); feats = d["feats"]
tr, va = d["tr"][d["tr"].year == 2023].reset_index(drop=True), d["tr"][d["tr"].year == 2024].reset_index(drop=True)
rate_base, _ = P.predict_rate(line, cfg, tr, va, feats); rate_base = np.asarray(rate_base)
mc_va = np.asarray(BandedTweedieGLM(**cfg["glm"][line]).fit(tr[feats], tr.target / tr.weight, tr.weight).predict_rate(va[feats]))

dev = P.developed_table(line, cfg); f = [c for c in feats if c in dev.columns]
both = pd.concat([dev[f + ["target", "weight"]].assign(market="developed"),
                  tr[f + ["target", "weight"]].assign(market="emerging")], ignore_index=True)
g = BandedTweedieGLM(**PART1["glm"][line]).fit(both[f + ["market"]], both.target / both.weight, both.weight)
pr = lambda X: np.asarray(g.predict_rate(X[f].assign(market="emerging")))
lvl = tr.target.sum() / (pr(tr) * tr.weight).sum(); o_tr, o_va = pr(tr) * lvl, pr(va) * lvl
r_tr, w_tr = (tr.target / tr.weight).to_numpy(), tr.weight.to_numpy()
a = 10 / np.average(np.sqrt(o_tr), weights=w_tr)
c = BandedTweedieGLM(power=1.5, alpha=a, n_bands=10, drop=None, balance=True).fit(tr[SEL], r_tr / o_tr, w_tr * np.sqrt(o_tr))
k = tr.target.sum() / (o_tr * c.predict_rate(tr[SEL]) * w_tr).sum()
rates = {"current (emerging only)": rate_base,
         "combined (E24)": rate_base - mc_va + o_va,
         "E25 combined + 5 corrections": rate_base - mc_va + o_va * np.asarray(c.predict_rate(va[SEL])) * k}

w = va.weight.to_numpy(); act_all = va.target_all.to_numpy(); act_mod = va.target_modelled.to_numpy()
def d2(act, pv): return evaluate(pd.Series(act), pv, pd.Series(w), show=False)["D2"]
def deciles(rate, act):
    o = np.argsort(rate, kind="stable"); cw = np.cumsum(w[o]) / w.sum(); dec = np.minimum((cw * 10).astype(int), 9)
    pv, aa = (rate * w)[o], act[o]; out = []
    for kk in (0, 9):
        s = dec == kk; out += [aa[s].sum() / pv[s].sum(), (aa[s].sum() - aa[s].max()) / pv[s].sum()]
    return out
rows = []
for name, r in rates.items():
    lo, lo_x, hi, hi_x = deciles(r, act_mod)
    rows.append({"model": name, "D2 all perils": d2(act_all, (r + d["sep_rate"]) * w), "low (rule b)": lo, "top (rule b)": hi,
                 "low, largest removed": lo_x, "top, largest removed": hi_x})
t = pd.DataFrame(rows).set_index("model"); print(t.round(4).to_string())
assert abs(t.iloc[0, 0] - 0.0684) < 0.0005 and abs(t.iloc[1, 0] - 0.0950) < 0.0005, "does not reproduce E24"
b, n = t.iloc[0], t.iloc[2]; gain = n["D2 all perils"] - b["D2 all perils"]
tails = all(abs(n[kk] - 1) <= abs(b[kk] - 1) + 0.1 for kk in ["low, largest removed", "top, largest removed"])
print(f"\nE25 re-run: D2 {gain:+.4f} | tails (largest removed) {'ok' if tails else 'FAIL'} -> {'ADOPT' if gain >= 0.005 and tails else 'not adopted'}")

21:41:43 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
21:41:43 buildings: fire priced separately at 97.05 per exposure (developed)
21:41:44 buildings: own GLM for ['flood']; main GLM on the other perils
                              D2 all perils  low (rule b)  top (rule b)  low, largest removed  top, largest removed
model                                                                                                              
current (emerging only)              0.0684        0.6154        1.0215                0.5149                0.8889
combined (E24)                       0.0950        0.3698        0.9845                0.3200                0.8532
E25 combined + 5 corrections         0.0931        0.3924        0.9916                0.3458                0.8606

E25 re-run: D2 +0.0247 | tails (largest removed) FAIL -> not adopted


In [39]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
assert "**E25 re-run result:**" not in d and "Model frozen at v7" not in d
a = "  combined question (E24, E25 first run, this run), disclosed before running.\n"
assert d.count(a) == 1
d = d.replace(a, a + "- **E25 re-run result:** 2024 D2 0.0684 -> 0.0931 (+0.0247; combined alone 0.0950); lowest decile with the largest loss removed "
              "0.515 -> 0.346 (0.17 further from 1; limit 0.1); top 0.889 -> 0.861. **Not adopted.** The five corrections recover little of the "
              "safest-decile gap: its cause is not these features. Fifth time the same direction (E17b, E18, E24, E25 twice): buildings other perils "
              "stay on emerging data only (final).\n")
txt = """### Model frozen at v7 (Oct 24, 21:45) — decided by Woojin
- Part 2 final: **p2_v7_leverage (public 0.14336)**. After 21:45 no model change unless a definition error is found (as the excess and leverage fixes).
- Tonight, not adopted: E14 (motor freq x sev), E24 / E25 (buildings combined), E21 re-run (emerging relativities), E26 / v8 (new-business
  corrections; withdrawn by the guard, public 0.14077); diagnostic only: actual 2025 weather (public 0.14399, +0.0006, not used).
- Remaining days: tests for the new target code, bootstrap interval for the new-business load, Part 2 column of the assumptions register,
  office premium table (Krista), report, video, reproducibility check from a fresh clone.
"""
a2 = "## Assumptions register"; assert d.count(a2) == 1
d = d.replace(a2, txt + "\n" + a2)
p.write_text(d, encoding="utf-8"); print("recorded")

AssertionError: 

In [40]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
assert "**E25 re-run result:**" not in d and "Model frozen at v7" not in d
a = "combined question (E24, E25 first run, this run), disclosed before running.\n"
assert d.count(a) == 1
d = d.replace(a, a + "- **E25 re-run result:** 2024 D2 0.0684 -> 0.0931 (+0.0247; combined alone 0.0950); lowest decile with the largest loss removed "
              "0.515 -> 0.346 (0.17 further from 1; limit 0.1); top 0.889 -> 0.861. **Not adopted.** The five corrections recover little of the "
              "safest-decile gap: its cause is not these features. Fifth time the same direction (E17b, E18, E24, E25 twice): buildings other perils "
              "stay on emerging data only (final).\n")
txt = """### Model frozen at v7 (Oct 24, 21:45) — decided by Woojin
- Part 2 final: **p2_v7_leverage (public 0.14336)**. After 21:45 no model change unless a definition error is found (as the excess and leverage fixes).
- Tonight, not adopted: E14 (motor freq x sev), E24 / E25 (buildings combined), E21 re-run (emerging relativities), E26 / v8 (new-business
  corrections; withdrawn by the guard, public 0.14077); diagnostic only: actual 2025 weather (public 0.14399, +0.0006, not used).
- Remaining days: tests for the new target code, bootstrap interval for the new-business load, Part 2 column of the assumptions register,
  office premium table (Krista), report, video, reproducibility check from a fresh clone.
"""
a2 = "## Assumptions register"; assert d.count(a2) == 1
d = d.replace(a2, txt + "\n" + a2)
p.write_text(d, encoding="utf-8"); print("recorded")

recorded


In [41]:
import os, sys, subprocess; from pathlib import Path; os.chdir(Path.home()/"IAA")
f = Path("tests/test_target_net.py"); assert not f.exists(), "already exists"
f.write_text('''"""Tests for the Oct 24 target changes (Part 2): amounts net of excess, trend leverage, row completeness.

Same style as test_target.py: tiny datasets, the hand calculation next to every expected value.
Run from the repository root:  pytest -v
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

sys.path.append(str(Path(__file__).resolve().parents[1]))
import src.target as T  # noqa: E402
from src.target import ADJ, adjust_claims, build_model_target, load_claims, load_policies, match_claims  # noqa: E402


def policy(row_id, policy_id, start, end, vol=200, comp=100):
    days = (pd.Timestamp(end) - pd.Timestamp(start)).days + 1
    return dict(row_id=row_id, policy_id=policy_id, period_start=pd.Timestamp(start).date(),
                period_end=pd.Timestamp(end).date(), exposure=days / 365, voluntary_excess=vol, compulsory_excess=comp)


def claim(claim_id, policy_id, peril, loss, incurred, recoveries=0.0, report=None, line="motor"):
    return dict(claim_id=claim_id, policy_id=policy_id, line=line, peril=peril, loss_date=pd.Timestamp(loss).date(),
                report_date=pd.Timestamp(report or loss).date(), status="closed", paid_to_date=incurred,
                incurred_to_date=incurred, recoveries=recoveries)


def write_data(tmp_path, policies, claims, line="motor"):
    pd.DataFrame(policies).to_parquet(tmp_path / f"policies_{line}.parquet")
    pd.DataFrame(claims).to_parquet(tmp_path / "claims.parquet")
    return load_policies(line, "train", data_dir=tmp_path), load_claims(data_dir=tmp_path)


def test_net_amounts_are_not_reduced_by_the_excess_again(tmp_path):
    # amounts already net of excess: 1,000 incurred - 100 recovered = 900 (the 300 excess is not deducted again)
    pol, cl = write_data(tmp_path, [policy("R1", "P1", "2023-01-01", "2023-12-31")],
                         [claim("C1", "P1", "own_damage", "2023-05-01", 1000, 100)])
    m = match_claims(cl, pol, "motor", cap=None, amounts_net=True)
    assert m["net"].iloc[0] == pytest.approx(900)
    assert m["excess_in_amount"].iloc[0] == pytest.approx(300)


def test_leverage_net_amounts_match_the_ground_up_calculation(tmp_path):
    # own_damage 2024, trend 4%: ground-up 1,000 (net 700 + excess 300) x 1.04 = 1,040; minus 300 = 740,
    # the same as test_trend_applies_before_excess on gross amounts. Trending the net (700 x 1.04 = 728) is wrong.
    pol, cl = write_data(tmp_path, [policy("R1", "P1", "2024-01-01", "2024-12-31")],
                         [claim("C1", "P1", "own_damage", "2024-06-01", 700)])
    m = adjust_claims(match_claims(cl, pol, "motor", cap=None, amounts_net=True), ADJ["motor"])
    assert m["net_adj"].iloc[0] == pytest.approx(740)


def test_leverage_with_recoveries(tmp_path):
    # own_damage 2024, trend 4%: (700 + 300) x 1.04 = 1,040; - 300 excess = 740; - 100 x 1.04 recovered = 636
    pol, cl = write_data(tmp_path, [policy("R1", "P1", "2024-01-01", "2024-12-31")],
                         [claim("C1", "P1", "own_damage", "2024-06-01", 700, 100)])
    m = adjust_claims(match_claims(cl, pol, "motor", cap=None, amounts_net=True), ADJ["motor"])
    assert m["net_adj"].iloc[0] == pytest.approx(636)


def test_no_trend_leaves_net_amounts_unchanged(tmp_path):
    # bodily_injury 2024: trend 0%, no reform after 2023 -> the net amount 1,000 is unchanged
    pol, cl = write_data(tmp_path, [policy("R1", "P1", "2024-01-01", "2024-12-31")],
                         [claim("C1", "P1", "bodily_injury", "2024-03-01", 1000)])
    m = adjust_claims(match_claims(cl, pol, "motor", cap=None, amounts_net=True), ADJ["motor"])
    assert m["net_adj"].iloc[0] == pytest.approx(1000)


def test_row_completeness_keeps_the_year_total_and_lowers_late_rows(tmp_path, monkeypatch):
    # Two 2024 rows: one for the whole year, one starting 1 Dec. Claims are reported 60 days after loss, so the
    # December row is less complete at the valuation date (2024-12-30). Row completeness must keep the year's
    # total weight and give the December row a lower weight per exposure than the full-year row.
    pols = [policy("R1", "P1", "2024-01-01", "2024-12-31"), policy("R2", "P2", "2024-12-01", "2024-12-31")]
    cs = [claim(f"C{i}", "P1", "own_damage", f"2024-0{m}-01", 500, report=f"2024-0{m + 2}-01")
          for i, m in enumerate([1, 2, 3, 4, 5], 1)]
    pol, cl = write_data(tmp_path, pols, cs)
    monkeypatch.setattr(T, "AMOUNTS_NET", True)
    monkeypatch.setattr(T, "ROW_COMPLETENESS", False)
    base, _, _ = build_model_target(cl, pol, "motor", completeness_method="lynden_bell", years=[2024])
    monkeypatch.setattr(T, "ROW_COMPLETENESS", True)
    rowc, _, _ = build_model_target(cl, pol, "motor", completeness_method="lynden_bell", years=[2024])
    assert rowc["weight"].sum() == pytest.approx(base["weight"].sum())
    per_exp = (rowc["weight"] / rowc["exposure_clean"]).to_numpy()
    assert per_exp[1] < per_exp[0]
''', encoding="utf-8")
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-800:])

...................................................                      [100%]
51 passed in 1.71s




In [42]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### Independent target audit (Oct 24, 22:37) — results/target_audit/ [AI 114]
- Fresh chat (Claude Opus 5.5): stage 1 designed the target from the Overview and Dataset Description only; stage 2 compared with src/target.py,
  config and DECISIONS; stage 3 checked exposure, frequency, severity, a step waterfall and year / market comparisons on the data (PDF, 14 pages).
- **Definition matches on every core step**, found blind: amounts already net of excess (not deducted again; closed claims under 100 CU exist at
  excess 1,000: motor 6.7%, buildings 8%), recoveries, declined claims, date-based exposure, row totals, trend on ground-up loss with the excess
  fixed. Check: sum(target)/sum(weight) from build_model_target equals the direct calculation (494.7 / 441.7 / 97.2); frequency x severity
  = cost per exposure at every step.
- New level items, all judgement and within +-1% per line, partly offsetting; **no model change** (v7 stays):
  BI completeness by count vs amount (0.545 vs 0.518): motor +0.6% to +1.0% under; BI reports after 730 days (developed 1.45%): motor +0.4%;
  subsidence completeness from 52 claims (0.753 vs developed 0.662): buildings +0.5% (sensitivity up to +5%); open-claim recoveries not
  reflected (known): motor -0.6%. Recorded as known limitations.
- Cost concentration: renewal_decision = declined_by_insurer rows are 0.1-0.5% of exposure but 22% / 51% / 21% of 2025-level cost
  (motor / buildings / contents); a marker after large claims, not a termination (75-95% of these policies are in the test). Not a feature
  (leakage); used in the report to explain Tweedie D2 volatility and the uncertainty of two training years.
- 2023 buildings fires rise each quarter (9, 12, 13, 15): a regional cluster building up, not one event.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "Independent target audit" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8")
log = ("| 114 | H | Team asked a fresh chat to design the target blind from the competition documents, then audit our code and data. "
       "| Stage 1 blind design compared line by line with target.py; every step checked on the data (exposure, frequency, severity, waterfall). "
       "| Definition confirmed (second blind confirmation after AI 108); four small level items (+-1%, offsetting) recorded as limitations; "
       "cost concentration in declined_by_insurer rows found for the report. No model change. |\n")
q = Path("AI_USE_LOG.md"); lines = q.read_text(encoding="utf-8").splitlines(keepends=True)
assert not any(l.startswith("| 114 |") for l in lines)
last = max(i for i, l in enumerate(lines) if l.startswith("| 11"))
lines.insert(last + 1, log); q.write_text("".join(lines), encoding="utf-8"); print("recorded")

recorded
